In [0]:
-- Creamos el esquema (base de datos) para tu proyecto
CREATE SCHEMA IF NOT EXISTS datos_retail;
---- Le decimos a Databricks que use este catálogo por defecto
USE CATALOG workspace;
-- Le decimos a Databricks que use este esquema por defecto
USE datos_retail;

In [0]:
-- Creamos un volumen para guardar los archivos que escupa el scraper en Python
CREATE VOLUME IF NOT EXISTS raw_scraping_data;

In [0]:
-- Dimensión de los productos (Actualizada con nombre_producto)
CREATE TABLE IF NOT EXISTS datos_retail.dim_producto (
    id_producto BIGINT GENERATED ALWAYS AS IDENTITY, -- Autoincremental nativo
    sku_original STRING,
    competidor STRING,
    marca STRING,
    categoria STRING,
    nombre_producto STRING, -- 🚀 NUEVA: Para tooltips y detalle en Power BI
    url_producto STRING
) USING DELTA;

-- Tabla de hechos (Actualizada con cuotas)
CREATE TABLE IF NOT EXISTS datos_retail.fact_precios_diarios (
    id_hecho BIGINT GENERATED ALWAYS AS IDENTITY,
    id_producto BIGINT,
    fecha_extraccion DATE,
    precio_regular DOUBLE,
    precio_oferta DOUBLE,
    cuotas INT,             -- 🚀 NUEVA: Para analizar financiación
    hay_stock BOOLEAN
) USING DELTA
PARTITIONED BY (fecha_extraccion); -- Particionar por fecha optimiza las consultas de series de tiempo

In [0]:
%python
import os
import glob
from pyspark.sql.functions import col, regexp_extract
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, IntegerType
from delta.tables import DeltaTable # ⬅️ Importamos DeltaTable

# 1. Definir la ruta base
carpeta_data = "/Volumes/workspace/datos_retail/raw_scraping_data/" 

# 2. Definir el esquema estricto (una sola vez)
esquema_productos = StructType([
    StructField("sku_original", StringType(), True),
    StructField("competidor", StringType(), True),
    StructField("marca", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("nombre_producto", StringType(), True),
    StructField("url_producto", StringType(), True),
    StructField("precio_regular", DoubleType(), True),
    StructField("precio_oferta", DoubleType(), True),
    StructField("cuotas", IntegerType(), True),  
    StructField("hay_stock", BooleanType(), True)
])

# 3. Lectura MASIVA de todos los CSV a la vez
ruta_archivos = os.path.join(carpeta_data, "fravega_*.csv")

print("⏳ Leyendo todos los archivos CSV simultáneamente...")
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .schema(esquema_productos) \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("multiLine", "true") \
    .load(ruta_archivos)

# 4. Extraer la fecha dinámicamente usando _metadata.file_path (Compatible con Unity Catalog)
df_bronze = df_bronze.withColumn(
    "fecha_extraccion", 
    regexp_extract(col("_metadata.file_path"), r"fravega_(\d{4}-\d{2}-\d{2})\.csv", 1).cast("date")
)

# Eliminar duplicados absolutos
df_bronze = df_bronze.dropDuplicates(["sku_original", "competidor", "fecha_extraccion"])

# 5. UPSERT en Dimensión Producto usando la API DeltaTable (Sin Vistas Temporales ni SQL)
# Primero, garantizamos que para el MERGE solo quede la versión más reciente del producto
# si en los CSV cargados hay varias fechas para un mismo sku
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

ventana = Window.partitionBy("sku_original", "competidor").orderBy(col("fecha_extraccion").desc())
df_origen_dim = df_bronze.withColumn("rn", row_number().over(ventana)).filter(col("rn") == 1).drop("rn")

print("🔄 Ejecutando UPSERT en la tabla de dimensiones...")
delta_dim = DeltaTable.forName(spark, "datos_retail.dim_producto")

delta_dim.alias("destino").merge(
    df_origen_dim.alias("origen"),
    "destino.sku_original = origen.sku_original AND destino.competidor = origen.competidor"
).whenMatchedUpdate(set = {
    "marca": "origen.marca",
    "categoria": "origen.categoria",
    "url_producto": "origen.url_producto",
    "nombre_producto": "origen.nombre_producto"
}).whenNotMatchedInsert(values = {
    "sku_original": "origen.sku_original",
    "competidor": "origen.competidor",
    "marca": "origen.marca",
    "categoria": "origen.categoria",
    "url_producto": "origen.url_producto",
    "nombre_producto": "origen.nombre_producto"
}).execute()


# 6. INSERT en Tabla de Hechos
df_dim = spark.table("datos_retail.dim_producto")

df_fact = df_bronze.alias("bronze").join(
    df_dim.alias("dim"),
    on=["sku_original", "competidor"],
    how="inner"
).selectExpr(
    "dim.id_producto", 
    "bronze.fecha_extraccion", 
    "bronze.precio_regular",
    "bronze.precio_oferta",
    "bronze.hay_stock",
    "CAST(bronze.cuotas AS INT) AS cuotas"
)

# 7. Reemplazo dinámico manual 
fechas_procesadas = [row["fecha_extraccion"].strftime("%Y-%m-%d") 
                     for row in df_fact.select("fecha_extraccion").distinct().collect()]

if fechas_procesadas:
    fechas_str = ", ".join([f"'{f}'" for f in fechas_procesadas])
    condicion_replace = f"fecha_extraccion IN ({fechas_str})"

    print(f"📥 Insertando particiones en tabla de hechos: {condicion_replace}")
    df_fact.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("replaceWhere", condicion_replace) \
        .saveAsTable("datos_retail.fact_precios_diarios")
else:
    print("⚠️ No hay datos nuevos para procesar.")

print("\n🏆 ¡PIPELINE MASIVO COMPLETADO! Compatible con Unity Catalog.")

In [0]:
SELECT * FROM datos_retail.dim_producto;

In [0]:
SELECT * FROM datos_retail.fact_precios_diarios;